In [3]:
import pandas as pd
import numpy as np

df_reliable = pd.read_pickle("../processed/df_reliable.pkl")

print("Shape:", df_reliable.shape)
print("Date range:", df_reliable["report_date"].min(), "→", df_reliable["report_date"].max())
print("\nColumns:", df_reliable.columns.tolist())

Shape: (1650419, 28)
Date range: 2021-01-01 00:00:00 → 2026-09-13 00:00:00

Columns: ['market_name', 'commodity_name', 'report_date', 'state_id', 'state_name', 'market_id', 'commodity_id', 'total_arrivals', 'arrivals', 'unit_of_arrivals', 'variety', 'grade', 'min_price', 'max_price', 'modal_price', 'unit_of_price', 'year', 'month', 'day_of_week', 'week_of_year', 'days_since_last_report', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_7', 'price_roll_mean_7', 'price_roll_std_7', 'price_roll_mean_30']


In [4]:
DAILY_KEYS = [
    "market_name",
    "commodity_name",
    "report_date"
]

df_daily = (
    df_reliable
    .groupby(DAILY_KEYS, as_index=False)
    .agg(
        modal_price=("modal_price", "median"),
        min_price=("min_price", "median"),
        max_price=("max_price", "median"),
        arrivals=("arrivals", "sum"),
        variety_count=("variety", "nunique"),
        grade_count=("grade", "nunique")
    )
)

df_daily["report_date"] = pd.to_datetime(df_daily["report_date"])

print("Daily shape:", df_daily.shape)
print(
    "Date range:",
    df_daily["report_date"].min(),
    "→",
    df_daily["report_date"].max()
)

print("\nRows per market-commodity-date:",
      df_daily.duplicated(
          ["market_name", "commodity_name", "report_date"]
      ).sum())

Daily shape: (1615735, 9)
Date range: 2021-01-01 00:00:00 → 2026-09-13 00:00:00

Rows per market-commodity-date: 0


In [5]:
GROUP_KEYS = ["market_name", "commodity_name"]

df_7d = df_daily.copy()

for h in range(1, 8):
    target_lookup = df_daily[
        GROUP_KEYS + ["report_date", "modal_price"]
    ].rename(
        columns={
            "report_date": "target_date",
            "modal_price": f"target_price_{h}d"
        }
    )

    df_7d["target_date"] = (
        df_7d["report_date"] + pd.Timedelta(days=h)
    )

    df_7d = df_7d.merge(
        target_lookup,
        on=GROUP_KEYS + ["target_date"],
        how="left"
    )

    df_7d.drop(columns=["target_date"], inplace=True)

print("Shape:", df_7d.shape)

print("\nTarget availability:")
for h in range(1, 8):
    available = df_7d[f"target_price_{h}d"].notna().sum()
    print(
        f"t+{h}: {available:,} "
        f"({available / len(df_7d):.2%})"
    )

print("\nExample:")
print(
    df_7d[
        GROUP_KEYS +
        ["report_date"] +
        [f"target_price_{h}d" for h in range(1, 8)]
    ].head(10)
)

Shape: (1615735, 16)

Target availability:
t+1: 1,037,991 (64.24%)
t+2: 1,004,870 (62.19%)
t+3: 985,709 (61.01%)
t+4: 979,287 (60.61%)
t+5: 978,807 (60.58%)
t+6: 982,156 (60.79%)
t+7: 1,191,971 (73.77%)

Example:
    market_name commodity_name report_date  target_price_1d  target_price_2d  \
0  APMC Aatpadi    Pomegranate  2021-01-14          13500.0              NaN   
1  APMC Aatpadi    Pomegranate  2021-01-15              NaN          15000.0   
2  APMC Aatpadi    Pomegranate  2021-01-17          12000.0          12500.0   
3  APMC Aatpadi    Pomegranate  2021-01-18          12500.0          13500.0   
4  APMC Aatpadi    Pomegranate  2021-01-19          13500.0          13500.0   
5  APMC Aatpadi    Pomegranate  2021-01-20          13500.0          10500.0   
6  APMC Aatpadi    Pomegranate  2021-01-21          10500.0              NaN   
7  APMC Aatpadi    Pomegranate  2021-01-22              NaN          12500.0   
8  APMC Aatpadi    Pomegranate  2021-01-24              NaN        

In [6]:
GROUP_KEYS = ["market_name", "commodity_name"]

df_7d = df_7d.sort_values(
    GROUP_KEYS + ["report_date"]
).reset_index(drop=True)

grp = df_7d.groupby(GROUP_KEYS)["modal_price"]

df_7d["year"] = df_7d["report_date"].dt.year
df_7d["month"] = df_7d["report_date"].dt.month
df_7d["day_of_week"] = df_7d["report_date"].dt.dayofweek
df_7d["week_of_year"] = (
    df_7d["report_date"].dt.isocalendar().week.astype(int)
)

df_7d["days_since_last_report"] = (
    df_7d.groupby(GROUP_KEYS)["report_date"]
    .diff()
    .dt.days
)

df_7d["price_lag_1"] = grp.shift(1)
df_7d["price_lag_2"] = grp.shift(2)
df_7d["price_lag_3"] = grp.shift(3)
df_7d["price_lag_7"] = grp.shift(7)

df_7d["price_roll_mean_7"] = (
    grp.transform(
        lambda s: s.shift(1).rolling(7, min_periods=3).mean()
    )
)

df_7d["price_roll_std_7"] = (
    grp.transform(
        lambda s: s.shift(1).rolling(7, min_periods=3).std()
    )
)

df_7d["price_roll_mean_30"] = (
    grp.transform(
        lambda s: s.shift(1).rolling(30, min_periods=5).mean()
    )
)

print("Shape:", df_7d.shape)

print("\nNew feature columns:")
print([
    "year",
    "month",
    "day_of_week",
    "week_of_year",
    "days_since_last_report",
    "price_lag_1",
    "price_lag_2",
    "price_lag_3",
    "price_lag_7",
    "price_roll_mean_7",
    "price_roll_std_7",
    "price_roll_mean_30"
])

Shape: (1615735, 28)

New feature columns:
['year', 'month', 'day_of_week', 'week_of_year', 'days_since_last_report', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_7', 'price_roll_mean_7', 'price_roll_std_7', 'price_roll_mean_30']


In [7]:
# Same-day statistics across markets for each commodity
commodity_day = (
    df_7d
    .groupby(["commodity_name", "report_date"])
    .agg(
        same_day_market_count=("market_name", "nunique"),
        cross_market_mean=("modal_price", "mean"),
        same_day_price_median_all=("modal_price", "median")
    )
    .reset_index()
)

# Add same-day aggregate information
df_7d = df_7d.merge(
    commodity_day,
    on=["commodity_name", "report_date"],
    how="left"
)

# Other-market statistics
other_market_mean = (
    df_7d["cross_market_mean"] * df_7d["same_day_market_count"]
    - df_7d["modal_price"]
) / (
    df_7d["same_day_market_count"] - 1
)

df_7d["price_vs_market_median"] = (
    df_7d["modal_price"]
    / df_7d["same_day_price_median_all"]
)

df_7d["price_vs_other_market_mean"] = (
    df_7d["modal_price"]
    / other_market_mean
)

# Daily market movement relative to previous observation
df_7d["market_price_change"] = (
    df_7d["modal_price"]
    / df_7d["price_lag_1"]
    - 1
)

# Count other markets moving up/down
movement_stats = (
    df_7d
    .assign(
        market_up=df_7d["market_price_change"] > 0.02,
        market_down=df_7d["market_price_change"] < -0.02
    )
    .groupby(["commodity_name", "report_date"])
    .agg(
        total_markets=("market_name", "count"),
        total_up=("market_up", "sum"),
        total_down=("market_down", "sum")
    )
    .reset_index()
)

df_7d = df_7d.merge(
    movement_stats,
    on=["commodity_name", "report_date"],
    how="left"
)

# Exclude the current market from movement proportions
other_count = df_7d["total_markets"] - 1

df_7d["pct_other_markets_up"] = np.where(
    other_count > 0,
    (
        df_7d["total_up"]
        - (df_7d["market_price_change"] > 0.02).astype(int)
    ) / other_count,
    np.nan
)

df_7d["pct_other_markets_down"] = np.where(
    other_count > 0,
    (
        df_7d["total_down"]
        - (df_7d["market_price_change"] < -0.02).astype(int)
    ) / other_count,
    np.nan
)

df_7d.drop(
    columns=[
        "market_price_change",
        "total_markets",
        "total_up",
        "total_down"
    ],
    inplace=True
)

print("Shape:", df_7d.shape)

print("\nCross-market missing values:")
cross_cols = [
    "same_day_market_count",
    "cross_market_mean",
    "same_day_price_median_all",
    "price_vs_market_median",
    "price_vs_other_market_mean",
    "pct_other_markets_up",
    "pct_other_markets_down"
]

print(df_7d[cross_cols].isna().sum())

Shape: (1615735, 35)

Cross-market missing values:
same_day_market_count             0
cross_market_mean                 0
same_day_price_median_all         0
price_vs_market_median            0
price_vs_other_market_mean    48385
pct_other_markets_up          48385
pct_other_markets_down        48385
dtype: int64


In [8]:
for h in range(1, 8):
    df_7d[f"pct_change_{h}d"] = (
        (df_7d[f"target_price_{h}d"] - df_7d["modal_price"])
        / df_7d["modal_price"]
    )

    df_7d[f"direction_{h}d"] = pd.cut(
        df_7d[f"pct_change_{h}d"],
        bins=[-np.inf, -0.02, 0.02, np.inf],
        labels=["down", "flat", "up"]
    )

print("Direction distributions by horizon:\n")

for h in range(1, 8):
    direction_col = f"direction_{h}d"

    counts = df_7d[direction_col].value_counts()
    percentages = (
        df_7d[direction_col]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print(f"t+{h}")
    print(pd.DataFrame({
        "count": counts,
        "percentage": percentages
    }))
    print()

Direction distributions by horizon:

t+1
               count  percentage
direction_1d                    
flat          442906       42.67
down          299851       28.89
up            295234       28.44

t+2
               count  percentage
direction_2d                    
flat          386106       38.42
down          309417       30.79
up            309347       30.78

t+3
               count  percentage
direction_3d                    
flat          354547       35.97
up            317184       32.18
down          313978       31.85

t+4
               count  percentage
direction_4d                    
flat          334394       34.15
up            324778       33.16
down          320115       32.69

t+5
               count  percentage
direction_5d                    
up            332613       33.98
down          326766       33.38
flat          319428       32.63

t+6
               count  percentage
direction_6d                    
up            339066       34.52
down      

In [9]:
LOG_SOURCE_COLUMNS = [
    "modal_price",
    "min_price",
    "max_price",
    "price_lag_1",
    "price_lag_2",
    "price_lag_3",
    "price_lag_7",
    "price_roll_mean_7",
    "price_roll_std_7",
    "price_roll_mean_30"
]

for col in LOG_SOURCE_COLUMNS:
    df_7d[f"log_{col}"] = np.log1p(
        df_7d[col].clip(lower=0)
    )

LOG_FEATURES = [
    f"log_{col}"
    for col in LOG_SOURCE_COLUMNS
]

print("Created log features:")
print(LOG_FEATURES)

print("\nShape:", df_7d.shape)

print("\nMissing values:")
print(df_7d[LOG_FEATURES].isna().sum())

Created log features:
['log_modal_price', 'log_min_price', 'log_max_price', 'log_price_lag_1', 'log_price_lag_2', 'log_price_lag_3', 'log_price_lag_7', 'log_price_roll_mean_7', 'log_price_roll_std_7', 'log_price_roll_mean_30']

Shape: (1615735, 59)

Missing values:
log_modal_price               0
log_min_price                 0
log_max_price                 0
log_price_lag_1            1751
log_price_lag_2            3502
log_price_lag_3            5253
log_price_lag_7           12257
log_price_roll_mean_7      5253
log_price_roll_std_7       5253
log_price_roll_mean_30     8755
dtype: int64


In [10]:
split_train_end = pd.Timestamp("2025-06-30")
split_val_end = pd.Timestamp("2025-12-31")

train_7d = df_7d[
    df_7d["report_date"] <= split_train_end
].copy()

val_7d = df_7d[
    (df_7d["report_date"] > split_train_end) &
    (df_7d["report_date"] <= split_val_end)
].copy()

test_7d = df_7d[
    df_7d["report_date"] > split_val_end
].copy()

print("Train:", train_7d.shape)
print("Val  :", val_7d.shape)
print("Test :", test_7d.shape)

print("\nTrain dates:")
print(train_7d["report_date"].min(), "→", train_7d["report_date"].max())

print("\nValidation dates:")
print(val_7d["report_date"].min(), "→", val_7d["report_date"].max())

print("\nTest dates:")
print(test_7d["report_date"].min(), "→", test_7d["report_date"].max())

Train: (1238199, 59)
Val  : (150063, 59)
Test : (227473, 59)

Train dates:
2021-01-01 00:00:00 → 2025-06-30 00:00:00

Validation dates:
2025-07-01 00:00:00 → 2025-12-31 00:00:00

Test dates:
2026-01-01 00:00:00 → 2026-09-13 00:00:00


In [11]:
LOG_SOURCE_COLUMNS = [
    "modal_price",
    "min_price",
    "max_price",
    "price_lag_1",
    "price_lag_2",
    "price_lag_3",
    "price_lag_7",
    "price_roll_mean_7",
    "price_roll_std_7",
    "price_roll_mean_30"
]

for col in LOG_SOURCE_COLUMNS:
    df_7d[f"log_{col}"] = np.log1p(
        df_7d[col].clip(lower=0)
    )

LOG_FEATURES = [
    f"log_{col}"
    for col in LOG_SOURCE_COLUMNS
]

print("Created log features:")
print(LOG_FEATURES)

print("\nShape:", df_7d.shape)

print("\nMissing values:")
print(df_7d[LOG_FEATURES].isna().sum())

Created log features:
['log_modal_price', 'log_min_price', 'log_max_price', 'log_price_lag_1', 'log_price_lag_2', 'log_price_lag_3', 'log_price_lag_7', 'log_price_roll_mean_7', 'log_price_roll_std_7', 'log_price_roll_mean_30']

Shape: (1615735, 59)

Missing values:
log_modal_price               0
log_min_price                 0
log_max_price                 0
log_price_lag_1            1751
log_price_lag_2            3502
log_price_lag_3            5253
log_price_lag_7           12257
log_price_roll_mean_7      5253
log_price_roll_std_7       5253
log_price_roll_mean_30     8755
dtype: int64


In [12]:
CROSS_FEATURES = [
    "same_day_market_count",
    "cross_market_mean",
    "same_day_price_median_all",
    "price_vs_market_median",
    "price_vs_other_market_mean",
    "pct_other_markets_up",
    "pct_other_markets_down"
]

BASE_FEATURES = [
    "market_name",
    "commodity_name",
    "arrivals",
    "days_since_last_report",
    "year",
    "month",
    "day_of_week",
    "week_of_year"
]

REG_FEATURES_7D = (
    BASE_FEATURES
    + LOG_FEATURES
    + CROSS_FEATURES
)

print("Number of regression features:", len(REG_FEATURES_7D))
print("\nFeatures:")
for i, feature in enumerate(REG_FEATURES_7D, 1):
    print(f"{i:2}. {feature}")

Number of regression features: 25

Features:
 1. market_name
 2. commodity_name
 3. arrivals
 4. days_since_last_report
 5. year
 6. month
 7. day_of_week
 8. week_of_year
 9. log_modal_price
10. log_min_price
11. log_max_price
12. log_price_lag_1
13. log_price_lag_2
14. log_price_lag_3
15. log_price_lag_7
16. log_price_roll_mean_7
17. log_price_roll_std_7
18. log_price_roll_mean_30
19. same_day_market_count
20. cross_market_mean
21. same_day_price_median_all
22. price_vs_market_median
23. price_vs_other_market_mean
24. pct_other_markets_up
25. pct_other_markets_down


In [13]:
TARGET = "target_price_1d"

train_reg_1d = train_7d.dropna(
    subset=[TARGET] + REG_FEATURES_7D
).copy()

val_reg_1d = val_7d.dropna(
    subset=[TARGET] + REG_FEATURES_7D
).copy()

test_reg_1d = test_7d.dropna(
    subset=[TARGET] + REG_FEATURES_7D
).copy()

# Convert categoricals consistently
for df_part in [train_reg_1d, val_reg_1d, test_reg_1d]:
    df_part["market_name"] = df_part["market_name"].astype("category")
    df_part["commodity_name"] = df_part["commodity_name"].astype("category")

X_train_1d = train_reg_1d[REG_FEATURES_7D]
y_train_1d = train_reg_1d[TARGET]

X_val_1d = val_reg_1d[REG_FEATURES_7D]
y_val_1d = val_reg_1d[TARGET]

X_test_1d = test_reg_1d[REG_FEATURES_7D]
y_test_1d = test_reg_1d[TARGET]

print("Train:", X_train_1d.shape)
print("Val  :", X_val_1d.shape)
print("Test :", X_test_1d.shape)

print("\nTarget statistics:")
print(y_train_1d.describe())

Train: (762132, 25)
Val  : (93378, 25)
Test : (146366, 25)

Target statistics:
count    762132.000000
mean       3278.275408
std        2703.728665
min           1.000000
25%        1500.000000
50%        2500.000000
75%        4500.000000
max       55000.000000
Name: target_price_1d, dtype: float64


In [14]:
TARGET = "target_price_1d"

train_reg_1d = train_7d.dropna(
    subset=[TARGET] + REG_FEATURES_7D
).copy()

val_reg_1d = val_7d.dropna(
    subset=[TARGET] + REG_FEATURES_7D
).copy()

test_reg_1d = test_7d.dropna(
    subset=[TARGET] + REG_FEATURES_7D
).copy()

# Keep categorical columns consistent
for df_part in [train_reg_1d, val_reg_1d, test_reg_1d]:
    df_part["market_name"] = df_part["market_name"].astype("category")
    df_part["commodity_name"] = df_part["commodity_name"].astype("category")

X_train_1d = train_reg_1d[REG_FEATURES_7D]
y_train_1d = train_reg_1d[TARGET]

X_val_1d = val_reg_1d[REG_FEATURES_7D]
y_val_1d = val_reg_1d[TARGET]

X_test_1d = test_reg_1d[REG_FEATURES_7D]
y_test_1d = test_reg_1d[TARGET]

print("X_train:", X_train_1d.shape)
print("X_val  :", X_val_1d.shape)
print("X_test :", X_test_1d.shape)

print("\ny_train:", y_train_1d.shape)
print("y_val  :", y_val_1d.shape)
print("y_test :", y_test_1d.shape)

X_train: (762132, 25)
X_val  : (93378, 25)
X_test : (146366, 25)

y_train: (762132,)
y_val  : (93378,)
y_test : (146366,)


In [15]:
import lightgbm as lgb
from lightgbm import LGBMRegressor

regressor_1d = LGBMRegressor(
    objective="regression",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    min_data_in_leaf=100,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

regressor_1d.fit(
    X_train_1d,
    np.log1p(y_train_1d),
    categorical_feature=["market_name", "commodity_name"],
    eval_set=[(X_val_1d, np.log1p(y_val_1d))],
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100)
    ]
)

print("Best iteration:", regressor_1d.best_iteration_)

[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100


c:\Users\hp\anaconda3\envs\myenv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.060420 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4945
[LightGBM] [Info] Number of data points in the train set: 762132, number of used features: 25
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Start training from score 7.692303
Training until validation scores don't improve for 100 rounds
[100]	valid_0's l2: 0.0278297
[200]	valid_0's l2: 0.0246319
[300]	valid_0's l2: 0.0244439
[400]	valid_0's l2: 0.0243649
[500]	valid_0's l2: 0.0243076
[600]	valid_0's l2: 0.0242617
[700]	valid_0's l2: 0.0242314
[800]	valid_0's l2: 0.0242012
[900]	valid_0's l2: 0.0241876
[1000]	valid_0's l2: 0.0241717
[1100]	valid_0's l2: 0.0241665
[1200]	valid_0'

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predict in log space
pred_log_1d = regressor_1d.predict(X_test_1d)

# Convert back to actual rupee prices
pred_price_1d = np.expm1(pred_log_1d)

mae_1d = mean_absolute_error(
    y_test_1d,
    pred_price_1d
)

rmse_1d = np.sqrt(
    mean_squared_error(
        y_test_1d,
        pred_price_1d
    )
)

smape_1d = (
    100
    * np.mean(
        2 * np.abs(pred_price_1d - y_test_1d)
        / (
            np.abs(y_test_1d)
            + np.abs(pred_price_1d)
            + 1e-8
        )
    )
)

# Naive baseline: tomorrow's price = today's price
naive_1d = test_reg_1d["modal_price"].values

naive_mae_1d = mean_absolute_error(
    y_test_1d,
    naive_1d
)

naive_rmse_1d = np.sqrt(
    mean_squared_error(
        y_test_1d,
        naive_1d
    )
)

naive_smape_1d = (
    100
    * np.mean(
        2 * np.abs(naive_1d - y_test_1d)
        / (
            np.abs(naive_1d)
            + np.abs(y_test_1d)
            + 1e-8
        )
    )
)

print("t+1 Cross-market model:")
print(f"MAE  : ₹{mae_1d:,.2f}")
print(f"RMSE : ₹{rmse_1d:,.2f}")
print(f"sMAPE: {smape_1d:.2f}%")

print("\nt+1 Naive baseline:")
print(f"MAE  : ₹{naive_mae_1d:,.2f}")
print(f"RMSE : ₹{naive_rmse_1d:,.2f}")
print(f"sMAPE: {naive_smape_1d:.2f}%")

[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
t+1 Cross-market model:
MAE  : ₹274.92
RMSE : ₹523.27
sMAPE: 8.91%

t+1 Naive baseline:
MAE  : ₹281.07
RMSE : ₹578.21
sMAPE: 9.25%


In [17]:
def prepare_regression_horizon(h):
    target = f"target_price_{h}d"

    train_part = train_7d.dropna(
        subset=[target] + REG_FEATURES_7D
    ).copy()

    val_part = val_7d.dropna(
        subset=[target] + REG_FEATURES_7D
    ).copy()

    test_part = test_7d.dropna(
        subset=[target] + REG_FEATURES_7D
    ).copy()

    for df_part in [train_part, val_part, test_part]:
        df_part["market_name"] = df_part["market_name"].astype("category")
        df_part["commodity_name"] = df_part["commodity_name"].astype("category")

    return (
        train_part,
        val_part,
        test_part,
        train_part[REG_FEATURES_7D],
        train_part[target],
        val_part[REG_FEATURES_7D],
        val_part[target],
        test_part[REG_FEATURES_7D],
        test_part[target]
    )


print("Function created successfully.")

Function created successfully.


In [18]:
# Prepare t+2 data
(
    train_2d,
    val_2d,
    test_2d,
    X_train_2d,
    y_train_2d,
    X_val_2d,
    y_val_2d,
    X_test_2d,
    y_test_2d
) = prepare_regression_horizon(2)

# Train t+2 model
regressor_2d = LGBMRegressor(
    objective="regression",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    min_data_in_leaf=100,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

regressor_2d.fit(
    X_train_2d,
    np.log1p(y_train_2d),
    categorical_feature=["market_name", "commodity_name"],
    eval_set=[(X_val_2d, np.log1p(y_val_2d))],
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(200)
    ]
)

print("Best iteration:", regressor_2d.best_iteration_)

c:\Users\hp\anaconda3\envs\myenv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023727 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4933
[LightGBM] [Info] Number of data points in the train set: 739710, number of used features: 25
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Start training from score 7.682674
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l2: 0.0292892
[400]	valid_0's l2: 0.0289601
[600]	valid_0's l2: 0.0288433
[800]	valid_0's l2: 0.0287703
[1000]	valid_0's l2: 0.0286979
[1200]	v

In [19]:
# Predict in log space
pred_log_2d = regressor_2d.predict(X_test_2d)

# Convert back to actual prices
pred_price_2d = np.expm1(pred_log_2d)

mae_2d = mean_absolute_error(
    y_test_2d,
    pred_price_2d
)

rmse_2d = np.sqrt(
    mean_squared_error(
        y_test_2d,
        pred_price_2d
    )
)

smape_2d = (
    100
    * np.mean(
        2 * np.abs(pred_price_2d - y_test_2d)
        / (
            np.abs(y_test_2d)
            + np.abs(pred_price_2d)
            + 1e-8
        )
    )
)

# Naive baseline: today's price = t+2 forecast
naive_2d = test_2d["modal_price"].values

naive_mae_2d = mean_absolute_error(
    y_test_2d,
    naive_2d
)

naive_rmse_2d = np.sqrt(
    mean_squared_error(
        y_test_2d,
        naive_2d
    )
)

naive_smape_2d = (
    100
    * np.mean(
        2 * np.abs(naive_2d - y_test_2d)
        / (
            np.abs(naive_2d)
            + np.abs(y_test_2d)
            + 1e-8
        )
    )
)

print("t+2 Cross-market model:")
print(f"MAE  : ₹{mae_2d:,.2f}")
print(f"RMSE : ₹{rmse_2d:,.2f}")
print(f"sMAPE: {smape_2d:.2f}%")

print("\nt+2 Naive baseline:")
print(f"MAE  : ₹{naive_mae_2d:,.2f}")
print(f"RMSE : ₹{naive_rmse_2d:,.2f}")
print(f"sMAPE: {naive_smape_2d:.2f}%")

[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
t+2 Cross-market model:
MAE  : ₹301.55
RMSE : ₹559.86
sMAPE: 9.88%

t+2 Naive baseline:
MAE  : ₹316.42
RMSE : ₹627.87
sMAPE: 10.48%


In [20]:
horizon_results = [
    {
        "horizon": 1,
        "mae": mae_1d,
        "rmse": rmse_1d,
        "smape": smape_1d,
        "naive_mae": naive_mae_1d,
        "naive_rmse": naive_rmse_1d,
        "naive_smape": naive_smape_1d
    },
    {
        "horizon": 2,
        "mae": mae_2d,
        "rmse": rmse_2d,
        "smape": smape_2d,
        "naive_mae": naive_mae_2d,
        "naive_rmse": naive_rmse_2d,
        "naive_smape": naive_smape_2d
    }
]

regressors_7d = {
    1: regressor_1d,
    2: regressor_2d
}

predictions_7d = {
    1: pred_price_1d,
    2: pred_price_2d
}

for h in range(3, 8):

    print(f"\n{'='*60}")
    print(f"Training t+{h}")
    print(f"{'='*60}")

    (
        train_h,
        val_h,
        test_h,
        X_train_h,
        y_train_h,
        X_val_h,
        y_val_h,
        X_test_h,
        y_test_h
    ) = prepare_regression_horizon(h)

    model_h = LGBMRegressor(
        objective="regression",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=63,
        min_data_in_leaf=100,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model_h.fit(
        X_train_h,
        np.log1p(y_train_h),
        categorical_feature=["market_name", "commodity_name"],
        eval_set=[(X_val_h, np.log1p(y_val_h))],
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(500)
        ]
    )

    pred_log_h = model_h.predict(X_test_h)
    pred_price_h = np.expm1(pred_log_h)

    mae_h = mean_absolute_error(y_test_h, pred_price_h)

    rmse_h = np.sqrt(
        mean_squared_error(y_test_h, pred_price_h)
    )

    smape_h = (
        100
        * np.mean(
            2 * np.abs(pred_price_h - y_test_h)
            / (
                np.abs(y_test_h)
                + np.abs(pred_price_h)
                + 1e-8
            )
        )
    )

    naive_h = test_h["modal_price"].values

    naive_mae_h = mean_absolute_error(
        y_test_h,
        naive_h
    )

    naive_rmse_h = np.sqrt(
        mean_squared_error(
            y_test_h,
            naive_h
        )
    )

    naive_smape_h = (
        100
        * np.mean(
            2 * np.abs(naive_h - y_test_h)
            / (
                np.abs(naive_h)
                + np.abs(y_test_h)
                + 1e-8
            )
        )
    )

    regressors_7d[h] = model_h
    predictions_7d[h] = pred_price_h

    horizon_results.append({
        "horizon": h,
        "mae": mae_h,
        "rmse": rmse_h,
        "smape": smape_h,
        "naive_mae": naive_mae_h,
        "naive_rmse": naive_rmse_h,
        "naive_smape": naive_smape_h,
        "best_iteration": model_h.best_iteration_
    })

    print(f"Best iteration: {model_h.best_iteration_}")
    print(f"Model MAE    : ₹{mae_h:,.2f}")
    print(f"Naive MAE    : ₹{naive_mae_h:,.2f}")
    print(f"Model RMSE   : ₹{rmse_h:,.2f}")
    print(f"Naive RMSE   : ₹{naive_rmse_h:,.2f}")
    print(f"Model sMAPE  : {smape_h:.2f}%")
    print(f"Naive sMAPE  : {naive_smape_h:.2f}%")

horizon_results_df = pd.DataFrame(horizon_results).sort_values("horizon")

print("\nFinal horizon comparison:")
print(horizon_results_df.to_string(index=False))


Training t+3


c:\Users\hp\anaconda3\envs\myenv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.124247 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4937
[LightGBM] [Info] Number of data points in the train set: 726812, number of used features: 25
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Start training from score 7.683627
Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.0325928
[1000]	valid_0's l2: 0.0323925
Early stopping, best iteration is:
[1299]	valid_0's l2: 0.0323478
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current 

c:\Users\hp\anaconda3\envs\myenv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012230 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4932
[LightGBM] [Info] Number of data points in the train set: 723229, number of used features: 25
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Start training from score 7.682401
Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.0349347
[1000]	valid_0's l2: 0.0347631
Early stopping, best iteration is:
[1043]	valid_0's l2: 0.0347608
[LightGBM] [Warning] min_data_in

...

[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014125 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4933
[LightGBM] [Info] Number of data points in the train set: 720845, number of used features: 25
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Start training from score 7.678429
Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.037748
[1000]	valid_0's l2: 0.0376015
Early stopping, best iteration is:
[941]	valid_0's l2: 0.0375951
[LightGBM] [Warning] min_data_in_l

c:\Users\hp\anaconda3\envs\myenv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014987 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4932
[LightGBM] [Info] Number of data points in the train set: 721558, number of used features: 25
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Start training from score 7.683907
Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.0404525
Early stopping, best iteration is:
[671]	valid_0's l2: 0.040369
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_sampl

c:\Users\hp\anaconda3\envs\myenv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016680 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4946
[LightGBM] [Info] Number of data points in the train set: 876294, number of used features: 25
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Start training from score 7.698591
Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.0426524
Early stopping, best iteration is:
[535]	valid_0's l2: 0.0426281
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samp

In [21]:
# Build an aligned prediction table using the same rows
# for which each horizon was evaluated.

forecast_path = None

for h in range(1, 8):

    # Recreate the exact test rows used for this horizon
    _, _, test_h, _, _, _, _, X_test_h, _ = prepare_regression_horizon(h)

    # Predict using the already-trained model
    pred_log_h = regressors_7d[h].predict(X_test_h)
    pred_price_h = np.expm1(pred_log_h)

    pred_h = test_h[
        [
            "market_name",
            "commodity_name",
            "report_date",
            "modal_price",
            f"target_price_{h}d"
        ]
    ].copy()

    pred_h[f"pred_price_{h}d"] = pred_price_h

    # Keep only the columns needed for merging
    pred_h = pred_h[
        [
            "market_name",
            "commodity_name",
            "report_date",
            "modal_price",
            f"target_price_{h}d",
            f"pred_price_{h}d"
        ]
    ]

    if forecast_path is None:
        forecast_path = pred_h
    else:
        forecast_path = forecast_path.merge(
            pred_h,
            on=[
                "market_name",
                "commodity_name",
                "report_date",
                "modal_price"
            ],
            how="inner"
        )

print("Common 7-day path rows:", len(forecast_path))
print("\nColumns:")
print(forecast_path.columns.tolist())

print("\nFirst 5 rows:")
print(forecast_path.head())

[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
Common 7-day path rows: 11492

Columns:
['market_name', 'commodity_name', 'report_date', 'modal_price', 'target_price_1d', 'pred_pri

In [22]:
path_results = []

for h in range(1, 8):

    y_true = forecast_path[f"target_price_{h}d"]
    y_pred = forecast_path[f"pred_price_{h}d"]

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    smape = (
        100
        * np.mean(
            2 * np.abs(y_pred - y_true)
            / (
                np.abs(y_true)
                + np.abs(y_pred)
                + 1e-8
            )
        )
    )

    path_results.append({
        "horizon": h,
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": smape
    })

path_results_df = pd.DataFrame(path_results)

print(path_results_df.round(2).to_string(index=False))

print("\nAverage across all 7 horizons:")
print(f"MAE   : ₹{path_results_df['MAE'].mean():,.2f}")
print(f"RMSE  : ₹{path_results_df['RMSE'].mean():,.2f}")
print(f"sMAPE : {path_results_df['sMAPE'].mean():.2f}%")

 horizon    MAE   RMSE  sMAPE
       1 272.11 475.75  10.32
       2 299.55 508.56  11.36
       3 319.67 536.78  12.10
       4 333.82 551.44  12.70
       5 346.54 567.71  13.21
       6 357.90 584.27  13.54
       7 371.65 603.56  14.13

Average across all 7 horizons:
MAE   : ₹328.75
RMSE  : ₹546.87
sMAPE : 12.48%


In [23]:
price_cols = [
    f"pred_price_{h}d"
    for h in range(1, 8)
]

current_price = forecast_path["modal_price"]

forecast_path["pred_max_price"] = forecast_path[price_cols].max(axis=1)
forecast_path["pred_min_price"] = forecast_path[price_cols].min(axis=1)
forecast_path["pred_avg_price"] = forecast_path[price_cols].mean(axis=1)

forecast_path["pred_max_change_pct"] = (
    (forecast_path["pred_max_price"] - current_price)
    / current_price
) * 100

forecast_path["pred_min_change_pct"] = (
    (forecast_path["pred_min_price"] - current_price)
    / current_price
) * 100

forecast_path["pred_avg_change_pct"] = (
    (forecast_path["pred_avg_price"] - current_price)
    / current_price
) * 100

# Number of future days predicted above / below today's price
predicted_matrix = forecast_path[price_cols].to_numpy()

forecast_path["days_predicted_above"] = (
    predicted_matrix > current_price.to_numpy()[:, None]
).sum(axis=1)

forecast_path["days_predicted_below"] = (
    predicted_matrix < current_price.to_numpy()[:, None]
).sum(axis=1)

print(
    forecast_path[
        [
            "modal_price",
            "pred_max_price",
            "pred_min_price",
            "pred_avg_price",
            "pred_max_change_pct",
            "pred_min_change_pct",
            "pred_avg_change_pct",
            "days_predicted_above",
            "days_predicted_below"
        ]
    ].head()
)

   modal_price  pred_max_price  pred_min_price  pred_avg_price  \
0       2600.0     2609.611103     2574.353346     2592.410642   
1       2600.0     2609.940765     2566.302548     2586.878135   
2       2600.0     2610.516645     2573.754093     2589.771479   
3       2600.0     2617.205824     2544.846563     2581.222361   
4       2600.0     2594.025103     2572.721522     2586.655282   

   pred_max_change_pct  pred_min_change_pct  pred_avg_change_pct  \
0             0.369658            -0.986410            -0.291898   
1             0.382337            -1.296056            -0.504687   
2             0.404486            -1.009458            -0.393405   
3             0.661762            -2.121286            -0.722217   
4            -0.229804            -1.049172            -0.513258   

   days_predicted_above  days_predicted_below  
0                     2                     5  
1                     1                     6  
2                     1                     6  
3 

In [24]:
actual_price_cols = [
    f"target_price_{h}d"
    for h in range(1, 8)
]

actual_matrix = forecast_path[actual_price_cols].to_numpy()
current_price = forecast_path["modal_price"].to_numpy()

forecast_path["actual_max_price"] = actual_matrix.max(axis=1)
forecast_path["actual_min_price"] = actual_matrix.min(axis=1)
forecast_path["actual_avg_price"] = actual_matrix.mean(axis=1)

forecast_path["actual_max_change_pct"] = (
    (forecast_path["actual_max_price"] - current_price)
    / current_price
) * 100

forecast_path["actual_min_change_pct"] = (
    (forecast_path["actual_min_price"] - current_price)
    / current_price
) * 100

forecast_path["actual_avg_change_pct"] = (
    (forecast_path["actual_avg_price"] - current_price)
    / current_price
) * 100

forecast_path["actual_final_change_pct"] = (
    (forecast_path["target_price_7d"] - current_price)
    / current_price
) * 100

print(
    forecast_path[
        [
            "modal_price",
            "pred_max_change_pct",
            "pred_min_change_pct",
            "pred_avg_change_pct",
            "actual_max_change_pct",
            "actual_min_change_pct",
            "actual_avg_change_pct",
            "actual_final_change_pct"
        ]
    ].head(10)
)

   modal_price  pred_max_change_pct  pred_min_change_pct  pred_avg_change_pct  \
0       2600.0             0.369658            -0.986410            -0.291898   
1       2600.0             0.382337            -1.296056            -0.504687   
2       2600.0             0.404486            -1.009458            -0.393405   
3       2600.0             0.661762            -2.121286            -0.722217   
4       2600.0            -0.229804            -1.049172            -0.513258   
5       2600.0            -0.055548            -1.290866            -0.510894   
6       2600.0             0.512529            -0.635062            -0.146953   
7       2600.0             0.049688            -0.926150            -0.399546   
8       2600.0            -0.131668            -1.022326            -0.542571   
9       2600.0             0.183586            -1.069872            -0.518179   

   actual_max_change_pct  actual_min_change_pct  actual_avg_change_pct  \
0                    0.0          

In [25]:
validation_path = None

for h in range(1, 8):

    # Prepare validation rows for this horizon
    (
        train_h,
        val_h,
        test_h,
        X_train_h,
        y_train_h,
        X_val_h,
        y_val_h,
        X_test_h,
        y_test_h
    ) = prepare_regression_horizon(h)

    # Predict using the already-trained horizon model
    pred_log_h = regressors_7d[h].predict(X_val_h)
    pred_price_h = np.expm1(pred_log_h)

    pred_h = val_h[
        [
            "market_name",
            "commodity_name",
            "report_date",
            "modal_price",
            f"target_price_{h}d"
        ]
    ].copy()

    pred_h[f"pred_price_{h}d"] = pred_price_h

    pred_h = pred_h[
        [
            "market_name",
            "commodity_name",
            "report_date",
            "modal_price",
            f"target_price_{h}d",
            f"pred_price_{h}d"
        ]
    ]

    if validation_path is None:
        validation_path = pred_h
    else:
        validation_path = validation_path.merge(
            pred_h,
            on=[
                "market_name",
                "commodity_name",
                "report_date",
                "modal_price"
            ],
            how="inner"
        )

print("Common validation 7-day path rows:", len(validation_path))

print("\nColumns:")
print(validation_path.columns.tolist())

print("\nFirst 5 rows:")
print(validation_path.head())

[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
Common validation 7-day path rows: 6078

Columns:
['market_name', 'commodity_name', 'report_date', 'modal_price', 'target_price_1d',

In [26]:
actual_price_cols = [
    f"target_price_{h}d"
    for h in range(1, 8)
]

pred_price_cols = [
    f"pred_price_{h}d"
    for h in range(1, 8)
]

actual_matrix = validation_path[actual_price_cols].to_numpy()
pred_matrix = validation_path[pred_price_cols].to_numpy()

current_price = validation_path["modal_price"].to_numpy()

validation_path["actual_max_price"] = actual_matrix.max(axis=1)
validation_path["actual_min_price"] = actual_matrix.min(axis=1)
validation_path["actual_avg_price"] = actual_matrix.mean(axis=1)
validation_path["actual_final_price"] = validation_path["target_price_7d"]

validation_path["actual_max_change_pct"] = (
    (validation_path["actual_max_price"] - validation_path["modal_price"])
    / validation_path["modal_price"]
) * 100

validation_path["actual_min_change_pct"] = (
    (validation_path["actual_min_price"] - validation_path["modal_price"])
    / validation_path["modal_price"]
) * 100

validation_path["actual_avg_change_pct"] = (
    (validation_path["actual_avg_price"] - validation_path["modal_price"])
    / validation_path["modal_price"]
) * 100

validation_path["actual_final_change_pct"] = (
    (validation_path["actual_final_price"] - validation_path["modal_price"])
    / validation_path["modal_price"]
) * 100

validation_path["pred_max_price"] = pred_matrix.max(axis=1)
validation_path["pred_min_price"] = pred_matrix.min(axis=1)
validation_path["pred_avg_price"] = pred_matrix.mean(axis=1)

validation_path["pred_max_change_pct"] = (
    (validation_path["pred_max_price"] - validation_path["modal_price"])
    / validation_path["modal_price"]
) * 100

validation_path["pred_min_change_pct"] = (
    (validation_path["pred_min_price"] - validation_path["modal_price"])
    / validation_path["modal_price"]
) * 100

validation_path["pred_avg_change_pct"] = (
    (validation_path["pred_avg_price"] - validation_path["modal_price"])
    / validation_path["modal_price"]
) * 100

validation_path["pred_final_change_pct"] = (
    (validation_path["pred_price_7d"] - validation_path["modal_price"])
    / validation_path["modal_price"]
) * 100

validation_path["days_predicted_above"] = (
    pred_matrix > current_price[:, None]
).sum(axis=1)

validation_path["days_predicted_below"] = (
    pred_matrix < current_price[:, None]
).sum(axis=1)

print(
    validation_path[
        [
            "modal_price",
            "pred_max_change_pct",
            "pred_min_change_pct",
            "pred_avg_change_pct",
            "pred_final_change_pct",
            "actual_max_change_pct",
            "actual_min_change_pct",
            "actual_avg_change_pct",
            "actual_final_change_pct",
            "days_predicted_above",
            "days_predicted_below"
        ]
    ].head(10)
)

   modal_price  pred_max_change_pct  pred_min_change_pct  pred_avg_change_pct  \
0       2600.0             0.567429            -0.427989             0.080090   
1       2600.0             0.399666            -0.523162             0.045738   
2       2600.0             0.318203            -0.724404            -0.018307   
3       2600.0             0.388962            -0.414116            -0.198226   
4       2600.0             0.793047            -0.245293             0.204305   
5       2600.0             0.565889            -0.305779             0.114807   
6       2600.0             0.443251            -0.555261            -0.000500   
7       2600.0             0.447559            -0.346527             0.130098   
8       2600.0             0.369519            -0.459542             0.017546   
9       2600.0             0.359787            -0.573173            -0.022841   

   pred_final_change_pct  actual_max_change_pct  actual_min_change_pct  \
0              -0.427989          

In [27]:
print("Predicted final 7-day change:")
print(
    validation_path["pred_final_change_pct"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nActual final 7-day change:")
print(
    validation_path["actual_final_change_pct"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nCorrelation:")
print(
    validation_path[
        [
            "pred_final_change_pct",
            "actual_final_change_pct"
        ]
    ].corr().iloc[0, 1]
)

print("\nActual final direction:")
actual_direction = pd.cut(
    validation_path["actual_final_change_pct"],
    bins=[-np.inf, -2, 2, np.inf],
    labels=["down", "flat", "up"]
)

print(
    actual_direction
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Predicted final 7-day change:
count    6078.000000
mean        0.818148
std        17.849882
min       -54.052133
10%       -14.737047
25%        -6.904332
50%        -0.279084
75%         6.105586
90%        16.268540
max       708.849560
Name: pred_final_change_pct, dtype: float64

Actual final 7-day change:
count    6078.000000
mean       -0.307896
std        30.421805
min       -80.000000
10%       -30.000000
25%       -15.384615
50%         0.000000
75%         8.333333
90%        28.571429
max       809.090909
Name: actual_final_change_pct, dtype: float64

Correlation:
0.5923088174429584

Actual final direction:
actual_final_change_pct
down    40.47
up      30.27
flat    29.25
Name: proportion, dtype: float64


In [28]:
pred_direction = pd.Series(
    np.where(
        validation_path["pred_final_change_pct"] < -2,
        "down",
        np.where(
            validation_path["pred_final_change_pct"] > 2,
            "up",
            "flat"
        )
    ),
    index=validation_path.index
)

actual_direction = pd.cut(
    validation_path["actual_final_change_pct"],
    bins=[-np.inf, -2, 2, np.inf],
    labels=["down", "flat", "up"]
)

print("Correlation:")
print(
    validation_path[
        [
            "pred_final_change_pct",
            "actual_final_change_pct"
        ]
    ].corr().iloc[0, 1]
)

print("\nRegression-derived direction accuracy:")
print(
    (pred_direction.values == actual_direction.astype(str).values).mean()
)

print("\nDirection confusion matrix:")

print(
    pd.crosstab(
        actual_direction,
        pred_direction,
        rownames=["Actual"],
        colnames=["Predicted"]
    )
)

print("\nPredicted direction distribution:")
print(
    pred_direction.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Correlation:
0.5923088174429584

Regression-derived direction accuracy:
0.5519907864429089

Direction confusion matrix:
Predicted  down  flat    up
Actual                     
down       1500   341   619
flat        563   740   475
up          438   287  1115

Predicted direction distribution:
down    41.15
up      36.34
flat    22.51
Name: proportion, dtype: float64


In [29]:
# Simple path-based signals for validation

validation_path["signal"] = np.select(
    [
        (
            (validation_path["pred_final_change_pct"] > 2) &
            (validation_path["days_predicted_above"] >= 5)
        ),

        (
            (validation_path["pred_final_change_pct"] < -2) &
            (validation_path["days_predicted_below"] >= 5)
        )
    ],
    [
        "HOLD",
        "SELL"
    ],
    default="NO STRONG SIGNAL"
)

print("Validation signal counts:")
print(validation_path["signal"].value_counts())

print("\nSignal coverage:")
issued_val = validation_path[
    validation_path["signal"].isin(["HOLD", "SELL"])
]

print(
    f"{len(issued_val):,} / {len(validation_path):,}"
)

print(
    f"Coverage: "
    f"{len(issued_val) / len(validation_path):.2%}"
)

Validation signal counts:
signal
SELL                2367
HOLD                2059
NO STRONG SIGNAL    1652
Name: count, dtype: int64

Signal coverage:
4,426 / 6,078
Coverage: 72.82%


In [30]:
threshold_results = []

for threshold in [1, 2, 3, 4, 5, 7, 10]:

    signal = np.select(
        [
            validation_path["pred_final_change_pct"] > threshold,
            validation_path["pred_final_change_pct"] < -threshold
        ],
        [
            "HOLD",
            "SELL"
        ],
        default="NO STRONG SIGNAL"
    )

    issued = signal != "NO STRONG SIGNAL"

    if issued.sum() == 0:
        continue

    correct = (
        (
            (signal == "HOLD") &
            (validation_path["actual_final_change_pct"] > 2)
        )
        |
        (
            (signal == "SELL") &
            (validation_path["actual_final_change_pct"] < -2)
        )
    )

    threshold_results.append({
        "threshold_pct": threshold,
        "issued": issued.sum(),
        "coverage": issued.mean(),
        "accuracy": correct[issued].mean()
    })

threshold_results_df = pd.DataFrame(threshold_results)

print(
    threshold_results_df.to_string(index=False)
)

 threshold_pct  issued  coverage  accuracy
             1    5332  0.877262  0.521193
             2    4710  0.774926  0.554989
             3    4292  0.706153  0.567102
             4    3904  0.642317  0.582992
             5    3552  0.584403  0.596284
             7    2897  0.476637  0.617190
            10    2136  0.351431  0.650749


In [31]:
strong_threshold_results = []

for threshold in [5, 7, 10]:

    signal = np.select(
        [
            validation_path["pred_final_change_pct"] > threshold,
            validation_path["pred_final_change_pct"] < -threshold
        ],
        [
            "HOLD",
            "SELL"
        ],
        default="NO STRONG SIGNAL"
    )

    issued = signal != "NO STRONG SIGNAL"

    hold_mask = signal == "HOLD"
    sell_mask = signal == "SELL"

    hold_correct = (
        validation_path["actual_final_change_pct"] > 2
    )

    sell_correct = (
        validation_path["actual_final_change_pct"] < -2
    )

    strong_threshold_results.append({
        "threshold": threshold,
        "coverage": issued.mean(),
        "overall_accuracy": (
            (
                (hold_mask & hold_correct)
                |
                (sell_mask & sell_correct)
            )[issued]
        ).mean(),
        "hold_issued": hold_mask.sum(),
        "hold_accuracy": (
            hold_correct[hold_mask].mean()
            if hold_mask.sum() > 0 else np.nan
        ),
        "sell_issued": sell_mask.sum(),
        "sell_accuracy": (
            sell_correct[sell_mask].mean()
            if sell_mask.sum() > 0 else np.nan
        )
    })

strong_threshold_results_df = pd.DataFrame(
    strong_threshold_results
)

print(
    strong_threshold_results_df.to_string(index=False)
)

 threshold  coverage  overall_accuracy  hold_issued  hold_accuracy  sell_issued  sell_accuracy
         5  0.584403          0.596284         1683       0.536542         1869       0.650080
         7  0.476637          0.617190         1393       0.550610         1504       0.678856
        10  0.351431          0.650749         1064       0.585526         1072       0.715485


In [32]:
# Predicted 7-day change
forecast_path["pred_final_change_pct"] = (
    (
        forecast_path["pred_price_7d"]
        - forecast_path["modal_price"]
    )
    / forecast_path["modal_price"]
) * 100

# Actual 7-day change
forecast_path["actual_final_change_pct"] = (
    (
        forecast_path["target_price_7d"]
        - forecast_path["modal_price"]
    )
    / forecast_path["modal_price"]
) * 100

print(
    forecast_path[
        [
            "modal_price",
            "pred_price_7d",
            "target_price_7d",
            "pred_final_change_pct",
            "actual_final_change_pct"
        ]
    ].head()
)

   modal_price  pred_price_7d  target_price_7d  pred_final_change_pct  \
0       2600.0    2609.611103           2600.0               0.369658   
1       2600.0    2609.940765           2600.0               0.382337   
2       2600.0    2610.516645           2600.0               0.404486   
3       2600.0    2555.473991           2600.0              -1.712539   
4       2600.0    2594.025103           2600.0              -0.229804   

   actual_final_change_pct  
0                      0.0  
1                      0.0  
2                      0.0  
3                      0.0  
4                      0.0  


In [33]:
TEST_THRESHOLD = 10

forecast_path["final_signal"] = np.select(
    [
        forecast_path["pred_final_change_pct"] > TEST_THRESHOLD,
        forecast_path["pred_final_change_pct"] < -TEST_THRESHOLD
    ],
    [
        "HOLD",
        "SELL"
    ],
    default="NO STRONG SIGNAL"
)

test_issued = forecast_path[
    forecast_path["final_signal"].isin(["HOLD", "SELL"])
].copy()

test_correct = (
    (
        (test_issued["final_signal"] == "HOLD") &
        (test_issued["actual_final_change_pct"] > 2)
    )
    |
    (
        (test_issued["final_signal"] == "SELL") &
        (test_issued["actual_final_change_pct"] < -2)
    )
)

print("Final test decision results")
print("--------------------------------")

print("\nSignal counts:")
print(forecast_path["final_signal"].value_counts())

print("\nCoverage:")
print(f"{len(test_issued) / len(forecast_path):.2%}")

print("\nOverall decision accuracy:")
print(f"{test_correct.mean():.2%}")

hold_test = test_issued["final_signal"] == "HOLD"
sell_test = test_issued["final_signal"] == "SELL"

print("\nHOLD accuracy:")
print(f"{test_correct[hold_test].mean():.2%}")

print("\nSELL accuracy:")
print(f"{test_correct[sell_test].mean():.2%}")

print("\nCorrect decisions:")
print(test_correct.sum())

print("Issued decisions:")
print(len(test_issued))

Final test decision results
--------------------------------

Signal counts:
final_signal
NO STRONG SIGNAL    7700
HOLD                2255
SELL                1537
Name: count, dtype: int64

Coverage:
33.00%

Overall decision accuracy:
65.22%

HOLD accuracy:
62.75%

SELL accuracy:
68.84%

Correct decisions:
2473
Issued decisions:
3792


In [38]:
from pathlib import Path
import json

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the 7 LightGBM models
for h in range(1, 8):
    model_path = MODEL_DIR / f"lightgbm_7day_{h}d.txt"
    regressors_7d[h].booster_.save_model(str(model_path))
    print(f"Saved: {model_path}")

# Save the exact feature list
feature_path = MODEL_DIR / "feature_cols_7day.json"

with open(feature_path, "w", encoding="utf-8") as f:
    json.dump(REG_FEATURES_7D, f, indent=2)

print(f"\nSaved: {feature_path}")

# Save the decision threshold
decision_config = {
    "forecast_horizons": [1, 2, 3, 4, 5, 6, 7],
    "decision_threshold_pct": 10,
    "hold_rule": "predicted_7d_change_pct > +10%",
    "sell_rule": "predicted_7d_change_pct < -10%",
    "otherwise": "NO STRONG SIGNAL"
}

decision_path = MODEL_DIR / "decision_config_7day.json"

with open(decision_path, "w", encoding="utf-8") as f:
    json.dump(decision_config, f, indent=2)

print(f"Saved: {decision_path}")

Saved: ..\models\lightgbm_7day_1d.txt
Saved: ..\models\lightgbm_7day_2d.txt
Saved: ..\models\lightgbm_7day_3d.txt
Saved: ..\models\lightgbm_7day_4d.txt
Saved: ..\models\lightgbm_7day_5d.txt
Saved: ..\models\lightgbm_7day_6d.txt
Saved: ..\models\lightgbm_7day_7d.txt

Saved: ..\models\feature_cols_7day.json
Saved: ..\models\decision_config_7day.json


In [39]:
import lightgbm as lgb
from pathlib import Path

MODEL_DIR = Path("../models")

loaded_models = {}

for h in range(1, 8):
    model_path = MODEL_DIR / f"lightgbm_7day_{h}d.txt"

    model = lgb.Booster(model_file=str(model_path))
    loaded_models[h] = model

    print(
        f"t+{h}: loaded successfully | "
        f"trees = {model.num_trees()}"
    )

print("\nAll 7 models loaded successfully.")

t+1: loaded successfully | trees = 1902
t+2: loaded successfully | trees = 1703
t+3: loaded successfully | trees = 1299
t+4: loaded successfully | trees = 1043
t+5: loaded successfully | trees = 941
t+6: loaded successfully | trees = 671
t+7: loaded successfully | trees = 535

All 7 models loaded successfully.


In [40]:
# Compare in-memory model vs saved model for one horizon first
h = 7

# Use the existing test feature matrix
saved_pred_log = loaded_models[h].predict(X_test_h)
saved_pred_price = np.expm1(saved_pred_log)

memory_pred_price = predictions_7d[h]

print("Number of predictions:", len(saved_pred_price))

print("\nFirst 10 predictions:")
comparison = pd.DataFrame({
    "memory_model": memory_pred_price[:10],
    "saved_model": saved_pred_price[:10],
    "difference": (
        memory_pred_price[:10]
        - saved_pred_price[:10]
    )
})

print(comparison)

print("\nMaximum absolute difference:")
print(
    np.max(
        np.abs(
            memory_pred_price
            - saved_pred_price
        )
    )
)

Number of predictions: 168471

First 10 predictions:
   memory_model   saved_model  difference
0  12675.392563  12675.392563         0.0
1  11469.000288  11469.000288         0.0
2  10818.552939  10818.552939         0.0
3  10095.004424  10095.004424         0.0
4  10232.217493  10232.217493         0.0
5  10123.507563  10123.507563         0.0
6   9250.485815   9250.485815         0.0
7   9617.267812   9617.267812         0.0
8   9682.557798   9682.557798         0.0
9   9544.408230   9544.408230         0.0

Maximum absolute difference:
0.0


In [41]:
verification_results = []

for h in range(1, 8):
    # Recreate the exact test matrix for this horizon
    (
        _,
        _,
        test_h,
        _,
        _,
        _,
        _,
        X_test_h,
        _
    ) = prepare_regression_horizon(h)

    # Prediction from saved model
    saved_pred = np.expm1(
        loaded_models[h].predict(X_test_h)
    )

    # Prediction from in-memory model
    memory_pred = predictions_7d[h]

    max_diff = np.max(
        np.abs(saved_pred - memory_pred)
    )

    verification_results.append({
        "horizon": h,
        "rows": len(saved_pred),
        "max_difference": max_diff
    })

verification_df = pd.DataFrame(verification_results)

print(verification_df.to_string(index=False))

print(
    "\nAll identical:",
    (verification_df["max_difference"] == 0).all()
)

 horizon   rows  max_difference
       1 146366             0.0
       2 140249             0.0
       3 137174             0.0
       4 135992             0.0
       5 137341             0.0
       6 138824             0.0
       7 168471             0.0

All identical: True
